# 면접 코칭 LLM 평가 자동화 파이프라인 (Gemini)

## 평가 체계
```
테스트셋 (testset.json)
    │
    ├── RAG 검색 품질 평가    → Context Relevance, Faithfulness
    ├── 응답 품질 평가        → Gemini-as-judge (4개 기준, 각 1-5점)
    ├── 오류 분석             → 낮은 점수 케이스 패턴 분류
    └── 회귀 테스트           → 이전 실행 결과와 점수 비교
```

**API 키:** https://aistudio.google.com/apikey 에서 무료 발급  
**실행 전 필요:** `01_rag_build.ipynb` 먼저 실행해서 `chroma_db/` 생성

In [1]:
%pip install python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, time
from pathlib import Path
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# ─── 경로 설정 ────────────────────────────────────────────────
BASE        = Path(r'C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)')
CHROMA_DIR  = BASE / 'chroma_db'
EVAL_DIR    = BASE / 'eval'
RESULTS_DIR = EVAL_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# ─── .env 파일에서 API 키 로드 ───────────────────────────────
load_dotenv(BASE / '.env')          # 프로젝트 폴더의 .env 읽기
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')

if not GOOGLE_API_KEY or GOOGLE_API_KEY == '여기에_실제_키_입력':
    raise ValueError(
        '.env 파일에 GOOGLE_API_KEY를 입력하세요.\n'
        f'파일 위치: {BASE / ".env"}'
    )

FEEDBACK_MODEL = 'gemini-flash-lite-latest'   # 무료 1500회/일
JUDGE_MODEL    = 'gemini-flash-lite-latest'

print('설정 완료')

설정 완료


In [3]:
# ─── Gemini 클라이언트 초기화 + 재시도 래퍼 ─────────────────
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

def gemini_call(model, contents, config=None, max_retries=5):
    """503/429 오류 시 지수 백오프로 자동 재시도"""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model, contents=contents, config=config
            )
        except Exception as e:
            code = getattr(e, 'status_code', 0)
            if code in (503, 429) and attempt < max_retries - 1:
                wait = 30 if code == 429 else 2 ** attempt  # 429는 30초, 503은 지수 백오프
                print(f'  [{code}] {wait}초 후 재시도... ({attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise

# 연결 확인
test = gemini_call(FEEDBACK_MODEL, '안녕하세요. 한 문장으로만 답하세요.')
print('Gemini 연결 성공:', test.text[:60])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini 연결 성공: 안녕하세요, 무엇을 도와드릴까요?


---
## STEP 1 — 테스트셋 로드

In [4]:
with open(EVAL_DIR / 'testset.json', encoding='utf-8') as f:
    testset = json.load(f)

cases = testset['cases']
print(f'테스트 케이스: {len(cases)}개')

from collections import Counter
print(f'카테고리 분포: {dict(Counter(c["category"] for c in cases))}')
print(f'답변 품질 분포: {dict(Counter(c["answer_quality"] for c in cases))}')

테스트 케이스: 15개
카테고리 분포: {'자기소개': 3, '지원동기': 3, '장단점': 2, '퇴직사유': 2, '압박질문': 3, '인성': 2}
답변 품질 분포: {'bad': 9, 'good': 6}


---
## STEP 2 — RAG 검색 (ChromaDB)

In [5]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
collection = chroma_client.get_collection('interview_rag', embedding_function=ef)
print(f'RAG 로드 완료: {collection.count()}개 청크')

def retrieve(question, answer, n=3):
    query = f"{question} {answer[:100]}"
    results = collection.query(query_texts=[query], n_results=n)
    return results['documents'][0], [m['source'] for m in results['metadatas'][0]]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
c:\Users\82105\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3429.11it/s]


RAG 로드 완료: 786개 청크


---
## STEP 3 — LLM 피드백 생성 (Gemini)

In [ ]:
SYSTEM_PROMPT = """당신은 대기업 인사담당자 출신의 직설적인 면접 코치 '이형'이다.
10년간 수천 명을 면접했고, 지원자 답변을 들으면 합격/불합격이 바로 보인다.
친근한 말투를 쓰되 평가는 냉정하게 한다.
제공된 참고 자료를 반드시 근거로 활용해 구체적인 피드백을 준다.
반드시 아래 형식으로만 답한다:

[이형의 팩폭 한줄평]
한 문장으로 이 답변의 핵심 문제 또는 강점을 직격한다.

[이형의 시선]
면접관 관점에서 이 답변이 어떻게 들리는지, 왜 좋은지/나쁜지 구체적으로 분석한다. (3~5문장)

[이형의 합격 처방전]
1. 즉시 실천 가능한 구체적 개선 방법
2. 답변 구조/내용 개선 방법
3. 면접관에게 어필할 포인트"""

def generate_feedback(question, answer, contexts):
    context_text = '\n\n'.join(f'[참고{i+1}] {c}' for i, c in enumerate(contexts))
    prompt = f"""면접 질문: {question}

지원자 답변: {answer}

--- 참고 자료 (면접왕 이형 채널) ---
{context_text}

위 답변에 대해 이형 스타일로 피드백하라."""

    resp = gemini_call(
        FEEDBACK_MODEL, f'{SYSTEM_PROMPT}\n\n{prompt}',
        config=types.GenerateContentConfig(temperature=0.3, max_output_tokens=700)
    )
    return resp.text

print('피드백 생성 함수 준비 완료 (이형 스타일)')

In [7]:
# 전체 테스트셋 실행
raw_results = []

for i, case in enumerate(cases):
    print(f'[{i+1}/{len(cases)}] {case["id"]} - {case["category"]} ({case["answer_quality"]})')

    contexts, sources = retrieve(case['question'], case['candidate_answer'])
    feedback = generate_feedback(case['question'], case['candidate_answer'], contexts)

    raw_results.append({
        'id':                 case['id'],
        'category':           case['category'],
        'answer_quality':     case['answer_quality'],
        'question':           case['question'],
        'candidate_answer':   case['candidate_answer'],
        'retrieved_contexts': contexts,
        'retrieved_sources':  sources,
        'generated_feedback': feedback,
        'expected_keywords':  case['expected_feedback_keywords'],
        'expected_direction': case['expected_feedback_direction'],
    })
    time.sleep(5)  # 분당 15회 한도 대비 (5초 간격)

print(f'\n피드백 생성 완료: {len(raw_results)}개')

[1/15] q001 - 자기소개 (bad)


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[2/15] q002 - 자기소개 (good)
[3/15] q003 - 지원동기 (bad)
[4/15] q004 - 지원동기 (good)
[5/15] q005 - 장단점 (bad)
[6/15] q006 - 장단점 (good)
[7/15] q007 - 퇴직사유 (bad)
[8/15] q008 - 퇴직사유 (good)
[9/15] q009 - 압박질문 (bad)
[10/15] q010 - 압박질문 (good)
[11/15] q011 - 인성 (bad)
[12/15] q012 - 인성 (good)
[13/15] q013 - 자기소개 (bad)
[14/15] q014 - 지원동기 (bad)
[15/15] q015 - 압박질문 (bad)

피드백 생성 완료: 15개


---
## STEP 4 — RAG 품질 평가

In [8]:
RAG_JUDGE_PROMPT = """아래 정보를 보고 두 가지 점수를 JSON으로만 출력하세요.

면접 질문: {question}
지원자 답변: {answer}
검색된 참고 문서: {context}
생성된 피드백: {feedback}

평가 기준:
- context_relevance: 검색 문서가 질문/답변 평가에 얼마나 관련 있는가 (0.0~1.0)
- faithfulness: 피드백이 검색 문서의 내용을 근거로 작성되었는가 (0.0~1.0)

JSON만 출력:
{{"context_relevance": 0.0, "faithfulness": 0.0}}"""

def eval_rag_quality(result):
    prompt = RAG_JUDGE_PROMPT.format(
        question=result['question'],
        answer=result['candidate_answer'][:200],
        context=' | '.join(result['retrieved_contexts'][:2])[:400],
        feedback=result['generated_feedback'][:300]
    )
    resp = gemini_call(
        JUDGE_MODEL, prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json')
    )
    return json.loads(resp.text)

print('RAG 품질 평가 함수 준비 완료')

RAG 품질 평가 함수 준비 완료


---
## STEP 5 — 응답 품질 평가 (Gemini-as-judge)

In [ ]:
QUALITY_JUDGE_PROMPT = """이형 스타일 면접 피드백의 품질을 4가지 기준으로 평가하세요. 각 항목을 1~5점으로 채점하고, 이유를 한 줄로 작성하세요.

면접 질문: {question}
지원자 답변: {answer}
피드백: {feedback}
예상 방향: {expected_direction}

평가 기준:
- factpunch (팩폭 한줄평): [이형의 팩폭 한줄평] 섹션이 핵심을 한 문장으로 정확히 직격했는가
- analysis (시선 분석): [이형의 시선] 섹션이 면접관 관점으로 구체적·근거 있게 분석했는가
- prescription (처방전): [이형의 합격 처방전] 3가지가 즉시 실천 가능한 구체적 조언인가
- accuracy (정확성): 예상 피드백 방향과 실제 피드백 내용이 일치하는가

JSON만 출력 (설명 금지):
{{"factpunch": 0, "factpunch_reason": "",
  "analysis": 0, "analysis_reason": "",
  "prescription": 0, "prescription_reason": "",
  "accuracy": 0, "accuracy_reason": ""}}"""

def eval_response_quality(result):
    prompt = QUALITY_JUDGE_PROMPT.format(
        question=result['question'],
        answer=result['candidate_answer'][:200],
        feedback=result['generated_feedback'][:500],
        expected_direction=result['expected_direction']
    )
    resp = gemini_call(
        JUDGE_MODEL, prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json')
    )
    return json.loads(resp.text)

print('품질 평가 함수 준비 완료 (이형 섹션 기준)')

In [10]:
# 키워드 적중률 + ROUGE-L (로컬 계산, API 호출 없음)
from rouge_score import rouge_scorer
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def keyword_hit_rate(feedback, expected_keywords):
    hits = [kw for kw in expected_keywords if kw in feedback]
    return len(hits) / len(expected_keywords) if expected_keywords else 0.0, hits

def rougeL(feedback, expected_direction):
    return round(rouge.score(expected_direction, feedback)['rougeL'].fmeasure, 3)

print('보조 평가 함수 준비 완료')

보조 평가 함수 준비 완료


In [ ]:
# ─── 전체 평가 실행 ──────────────────────────────────────────
eval_results = []

for i, result in enumerate(raw_results):
    print(f'[{i+1}/{len(raw_results)}] {result["id"]} 평가 중...')

    rag_scores  = eval_rag_quality(result)
    time.sleep(5)
    qual_scores = eval_response_quality(result)
    kw_rate, kw_hits = keyword_hit_rate(result['generated_feedback'], result['expected_keywords'])
    rouge_l = rougeL(result['generated_feedback'], result['expected_direction'])

    # 이형 4개 섹션 점수 평균
    qual_avg = round((qual_scores.get('factpunch', 0) + qual_scores.get('analysis', 0) +
                      qual_scores.get('prescription', 0) + qual_scores.get('accuracy', 0)) / 4, 2)

    eval_results.append({
        'id':             result['id'],
        'category':       result['category'],
        'answer_quality': result['answer_quality'],
        'context_relevance': rag_scores.get('context_relevance', 0),
        'faithfulness':      rag_scores.get('faithfulness', 0),
        'factpunch':      qual_scores.get('factpunch', 0),
        'analysis':       qual_scores.get('analysis', 0),
        'prescription':   qual_scores.get('prescription', 0),
        'accuracy':       qual_scores.get('accuracy', 0),
        'quality_avg':    qual_avg,
        'keyword_hit_rate': round(kw_rate, 3),
        'keyword_hits':     kw_hits,
        'rouge_l':          rouge_l,
        'factpunch_reason':    qual_scores.get('factpunch_reason', ''),
        'analysis_reason':     qual_scores.get('analysis_reason', ''),
        'prescription_reason': qual_scores.get('prescription_reason', ''),
        'accuracy_reason':     qual_scores.get('accuracy_reason', ''),
        'question':          result['question'],
        'candidate_answer':  result['candidate_answer'],
        'generated_feedback':result['generated_feedback'],
        'retrieved_sources': result['retrieved_sources'],
    })
    time.sleep(5)

df = pd.DataFrame(eval_results)
print(f'\n평가 완료: {len(df)}개')

---
## STEP 6 — 결과 요약 & 오류 분석

In [ ]:
metrics = ['context_relevance', 'faithfulness',
           'factpunch', 'analysis', 'prescription', 'accuracy',
           'keyword_hit_rate', 'rouge_l']

print('=' * 60)
print('  전체 평균 점수')
print('=' * 60)
rag_metrics  = {'context_relevance', 'faithfulness', 'keyword_hit_rate', 'rouge_l'}
for m in metrics:
    val = df[m].mean()
    # RAG/비율 지표는 0~1, LLM 점수는 1~5 — 막대 스케일 통일
    bar_len = int(val * 5 if m in rag_metrics else val)
    bar = '█' * bar_len
    label_map = {'factpunch': '팩폭한줄평', 'analysis': '시선분석',
                 'prescription': '합격처방전', 'accuracy': '정확성'}
    label = label_map.get(m, m)
    print(f'  {label:<22} {val:.3f}  {bar}')

print('\n카테고리별 quality_avg:')
print(df.groupby('category')['quality_avg'].mean().round(2).to_string())

print('\ngood/bad 답변별 quality_avg:')
print(df.groupby('answer_quality')['quality_avg'].mean().round(2).to_string())

In [ ]:
# ─── 오류 분석 ───────────────────────────────────────────────
THRESHOLD = 3.0
low_cases = df[df['quality_avg'] < THRESHOLD].sort_values('quality_avg')
print(f'품질 임계값({THRESHOLD}) 미달: {len(low_cases)}개\n')

for _, row in low_cases.iterrows():
    print(f"[{row['id']}] {row['category']} / {row['answer_quality']} — avg {row['quality_avg']}")
    print(f"  Q: {row['question']}")
    if row['specificity'] < 3:    print(f"  ⚠ 구체성 낮음: {row['specificity_reason']}")
    if row['accuracy'] < 3:       print(f"  ⚠ 방향 오류:   {row['accuracy_reason']}")
    if row['faithfulness'] < 0.5: print(f"  ⚠ RAG 근거 부족 (faithfulness={row['faithfulness']:.2f})")
    if row['keyword_hit_rate'] < 0.3: print(f"  ⚠ 키워드 미적중 ({row['keyword_hit_rate']:.0%})")
    print()

In [ ]:
# ─── 오류 패턴 분류 ──────────────────────────────────────────
def classify_error(row):
    errors = []
    if row['factpunch'] < 3:          errors.append('팩폭_약함')
    if row['analysis'] < 3:           errors.append('분석_부족')
    if row['prescription'] < 3:       errors.append('처방전_부실')
    if row['accuracy'] < 3:           errors.append('방향_오류')
    if row['faithfulness'] < 0.5:     errors.append('RAG근거_부족')
    if row['keyword_hit_rate'] < 0.3: errors.append('키워드_미적중')
    return errors if errors else ['정상']

df['error_types'] = df.apply(classify_error, axis=1)

from collections import Counter
all_errors = [e for errors in df['error_types'] for e in errors]
print('오류 유형별 빈도:')
for etype, cnt in Counter(all_errors).most_common():
    print(f'  {etype:<18} {cnt:2}건  {"■" * cnt}')

---
## STEP 7 — 결과 저장 & 회귀 테스트

In [ ]:
run_id = datetime.now().strftime('%Y%m%d_%H%M')
result_path = RESULTS_DIR / f'eval_{run_id}.json'

summary = {
    'run_id':   run_id,
    'model':    FEEDBACK_MODEL,
    'n_cases':  len(df),
    'scores':   {m: round(float(df[m].mean()), 3) for m in metrics},
    'low_quality_count': int(len(low_cases)),
    'error_counts': dict(Counter(all_errors)),
    'per_case': eval_results
}

with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

df.to_csv(RESULTS_DIR / f'eval_{run_id}.csv', index=False, encoding='utf-8-sig')
print(f'결과 저장: {result_path}')

In [ ]:
# ─── 회귀 테스트 ─────────────────────────────────────────────
prev_files = sorted(RESULTS_DIR.glob('eval_*.json'))[:-1]

if not prev_files:
    print('이전 결과 없음 — 첫 실행입니다. 다음 실행부터 회귀 테스트가 동작합니다.')
else:
    with open(prev_files[-1], encoding='utf-8') as f:
        prev = json.load(f)

    print(f'비교: {prev["run_id"]}  →  {run_id}\n')
    print(f'{"지표":<22} {"이전":>8} {"현재":>8} {"변화":>8} 판정')
    print('-' * 58)

    REGRESSION_THRESHOLD = -0.1
    has_regression = False

    for m in metrics:
        prev_val = prev['scores'].get(m, 0)
        curr_val = summary['scores'].get(m, 0)
        delta    = curr_val - prev_val
        flag = ''
        if delta < REGRESSION_THRESHOLD:
            flag = '🔴 회귀!'
            has_regression = True
        elif delta > 0.05:
            flag = '🟢 개선'
        print(f'{m:<22} {prev_val:>8.3f} {curr_val:>8.3f} {delta:>+8.3f}  {flag}')

    print()
    if has_regression:
        print('⚠️  회귀 감지 — 프롬프트/모델 변경사항을 검토하세요.')
    else:
        print('✅ 회귀 없음 — 모든 지표 유지 또는 개선')

---
## STEP 8 — 개별 케이스 상세 확인

In [ ]:
def inspect(case_id):
    row = df[df['id'] == case_id].iloc[0]
    print(f'{"="*65}')
    print(f'ID: {row["id"]}  |  카테고리: {row["category"]}  |  답변품질: {row["answer_quality"]}')
    print(f'{"="*65}')
    print(f'[질문] {row["question"]}')
    print(f'[지원자 답변] {row["candidate_answer"]}')
    print(f'[검색 출처] {row["retrieved_sources"]}')
    print()
    print(f'[생성된 피드백]\n{row["generated_feedback"]}')
    print()
    print(f'─ RAG  | relevance: {row["context_relevance"]:.2f} | faithfulness: {row["faithfulness"]:.2f}')
    print(f'─ 품질 | 팩폭:{row["factpunch"]}  시선:{row["analysis"]}  처방전:{row["prescription"]}  정확성:{row["accuracy"]}  → avg: {row["quality_avg"]}')
    print(f'─ 지표 | 키워드: {row["keyword_hit_rate"]:.0%} {row["keyword_hits"]}  |  ROUGE-L: {row["rouge_l"]}')
    print(f'─ 오류 | {row["error_types"]}')
    if row['factpunch_reason']:    print(f'  팩폭   → {row["factpunch_reason"]}')
    if row['analysis_reason']:     print(f'  시선   → {row["analysis_reason"]}')
    if row['prescription_reason']: print(f'  처방전 → {row["prescription_reason"]}')
    if row['accuracy_reason']:     print(f'  정확성 → {row["accuracy_reason"]}')

# 가장 낮은 케이스 자동 확인
worst_id = df.nsmallest(1, 'quality_avg')['id'].values[0]
inspect(worst_id)